In [9]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

In [10]:
from typing import TypedDict
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel,Field
from langgraph.graph import END,StateGraph
from typing import List


#### Orchestrator-Worker Pattern

##### Structured Output

In [11]:
# dish schema for a single dish
class Dish(BaseModel):
    name:str=Field(
        description="Name of the dish (for example, Spaghetti Bolognese, Chicken Curry)"
    )
    ingredients:List[str]=Field(
        description="List of the ingredients needed for this dish, separated by commas"
    )
    location:str=Field(
        description="The cuisine or cultural origin of the dish(for example italy,indian,nepal)"
    )
    

In [12]:
# dish schema for for a list of Dish object
class Dishes(BaseModel):
    sections:List[Dish]=Field(
        description="A list of gorcery sections, one for each dish, with ingedients"
    )

In [23]:
# construct a prompt template
dish_prompt=ChatPromptTemplate.from_messages([(
    "system",
    "you are an assistant that generates a structured grocery list.\n\n"
    "the user wants to prepare the following meals: {meals}\n\n"
    "for each meals, return a section with:\n"
    "- the name of the dish\n"
    "- a comma-separated list of ingredients needed for that dish.\n"
    "- the cusine or cultural origin of the food"
)
    
])

In [25]:
from langchain_core.prompts import ChatPromptTemplate

dish_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant that generates grocery lists."
    ),
    (
        "human",
        """The user wants to prepare these meals:

{meals}

For each meal, return:
- Name of the dish
- Ingredients (comma-separated)
- Cuisine or cultural origin
"""
    )
])

In [38]:
# use LCEL to pipe the prompt to an LLM with a structured output of Dishes
planner_pipe = dish_prompt | llm # use llm.with_structured_output(Dishes)

# invoke the planner_pipe with example meals
response=planner_pipe.invoke({"meals":["carrot cake"]})

In [39]:
from pprint import pprint
pprint(response.content)

('**Dish Name:** Carrot Cake\n'
 '\n'
 '**Ingredients:** All-purpose flour, baking soda, baking powder, cinnamon, '
 'nutmeg, salt, granulated sugar, brown sugar, eggs, vegetable oil, vanilla '
 'extract, shredded carrots, crushed pineapple (optional), walnuts or pecans '
 '(optional), raisins (optional), cream cheese, butter, powdered sugar\n'
 '\n'
 '**Cuisine/Cultural Origin:** European/American — Carrot cake has roots in '
 'medieval European carrot puddings, but its modern frosted form became '
 'popular in the United States during the mid-20th century and is now a '
 'classic dessert in American cuisine.')
